# Task 1 — SFT transfer of a simple explanation style to Qwen3

This notebook fine-tunes exactly `Qwen/Qwen3-4B-Instruct-2507` with 4-bit QLoRA on
`data/kid_adult.jsonl`. Only `prompt` is used as the user message and only `kid` is
used as the assistant target. The `adult` field is never a training target.

The held-out `public_test_style.jsonl` is not opened until training has finished. It is
then loaded once into memory for deterministic generation and final measurement only.
It is never placed in `train_dataset`, and no public-test result is used to tune the
hyperparameters. There is no system prompt or instruction asking the model to simplify.

Reproducibility contract: seed 42, greedy decoding (`do_sample=False`, `num_beams=1`),
pinned Python packages, a pinned Hugging Face model commit, explicit assertions, and no
pre-trained adapter download. CUDA/bitsandbytes training is deterministic to the extent
supported by the same Colab T4 software/hardware stack; bitwise identity across different
GPU models or CUDA kernels is not claimed.

## 1. Install the pinned software stack

PyTorch and CUDA come from Colab and are deliberately not reinstalled. The versions below
are pinned rather than resolved from `latest`; `scikit-learn==1.7.2` matches the version
used to serialize `style_clf.pkl`.

In [ ]:
import subprocess
import sys

PINNED_PACKAGES = [
    "numpy==1.26.4",
    "transformers==4.57.1",
    "trl==0.24.0",
    "peft==0.17.1",
    "accelerate==1.10.1",
    "bitsandbytes==0.48.1",
    "datasets==4.1.1",
    "scikit-learn==1.7.2",
    "scipy==1.16.2",
    "pandas==2.3.3",
    "joblib==1.5.2",
    "sentencepiece==0.2.1",
    "tokenizers==0.22.1",
    "safetensors==0.6.2",
    "huggingface-hub==0.35.3",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", *PINNED_PACKAGES]
)
subprocess.check_call([sys.executable, "-m", "pip", "check"])
print("Installed pinned dependencies; Colab's PyTorch/CUDA installation was left untouched.")

## 2. Print versions and require a CUDA GPU

In [ ]:
import importlib.metadata as metadata
import platform
import subprocess

import torch

distributions = [
    "numpy", "transformers", "trl", "peft", "accelerate", "bitsandbytes",
    "datasets", "scikit-learn", "scipy", "pandas", "joblib", "sentencepiece",
    "tokenizers", "safetensors", "huggingface-hub",
]
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
for distribution in distributions:
    print(f"{distribution}: {metadata.version(distribution)}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is required. In Colab select Runtime -> Change runtime type -> T4 GPU, "
        "then run the notebook again from the first cell."
    )
props = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print(f"GPU memory: {props.total_memory / 2**30:.2f} GiB")
subprocess.run(["nvidia-smi"], check=True)

## 3. One configuration and all random seeds

`REPO_URL` is intentionally empty because this local repository currently has no public
HTTPS remote. When a public remote exists, record its clone URL here; the bootstrap cell
will then clone it automatically when the notebook is opened directly in a clean Colab.
In an already cloned checkout, no URL or path edit is needed.

In [ ]:
import json
import os
import random
from pathlib import Path

import numpy as np
from transformers import set_seed

CONFIG = {
    "repo_url": "",
    "repo_branch": "main",
    "model_id": "Qwen/Qwen3-4B-Instruct-2507",
    "model_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
    "seed": 42,
    "num_train_epochs": 2,
    "learning_rate": 1e-4,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "max_length": 512,
    "warmup_ratio": 0.03,
    "lr_scheduler_type": "cosine",
    "optim": "paged_adamw_8bit",
    "weight_decay": 0.0,
    "max_grad_norm": 0.3,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    "generation_max_new_tokens": 192,
}
SEED = CONFIG["seed"]
assert SEED == 42

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if hasattr(torch.backends.cuda.matmul, "allow_tf32"):
    torch.backends.cuda.matmul.allow_tf32 = False

print(json.dumps(CONFIG, ensure_ascii=False, indent=2))
print("Effective train batch size:", CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"])

## 4. Locate or clone the repository without interactive file uploads

In [ ]:
import shutil

REQUIRED_RELATIVE_PATHS = [
    Path("data/kid_adult.jsonl"),
    Path("data/public_test_style.jsonl"),
    Path("metrics/style_clf.pkl"),
]

def find_repo_root(start: Path) -> Path | None:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if all((candidate / relative).is_file() for relative in REQUIRED_RELATIVE_PATHS):
            return candidate
    return None

repo_root = find_repo_root(Path.cwd())
if repo_root is None:
    repo_url = CONFIG["repo_url"].strip()
    if not repo_url.startswith("https://"):
        raise RuntimeError(
            "The repository files are not present in this runtime and CONFIG['repo_url'] "
            "does not contain a public HTTPS clone URL. The local repository had no remote "
            "when this notebook was generated, so automatic clean-Colab bootstrap cannot be "
            "claimed yet. Add a public remote, record it in CONFIG, commit, and reopen in Colab."
        )
    clone_dir = Path("/content/task1-sft-repo")
    if clone_dir.exists():
        shutil.rmtree(clone_dir)
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "--branch", CONFIG["repo_branch"], repo_url, str(clone_dir)]
    )
    repo_root = find_repo_root(clone_dir)

assert repo_root is not None
assert all((repo_root / relative).is_file() for relative in REQUIRED_RELATIVE_PATHS)
os.chdir(repo_root)
print("Repository root:", repo_root)

## 5. Load and validate training data only

The held-out file is deliberately not parsed in this section. The training dataset is
reconstructed with only `prompt` and `kid`, so neither `adult` nor any public-test field can
become a target accidentally.

In [ ]:
import hashlib

from datasets import Dataset

TRAIN_PATH = repo_root / "data" / "kid_adult.jsonl"
PUBLIC_PATH = repo_root / "data" / "public_test_style.jsonl"
STYLE_CLF_PATH = repo_root / "metrics" / "style_clf.pkl"

def read_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

train_rows = read_jsonl(TRAIN_PATH)
assert len(train_rows) == 1489, f"Expected 1489 training rows, got {len(train_rows)}"
assert all({"prompt", "kid", "adult"}.issubset(row) for row in train_rows)
assert all(isinstance(row["prompt"], str) and row["prompt"].strip() for row in train_rows)
assert all(isinstance(row["kid"], str) and row["kid"].strip() for row in train_rows)

train_examples = [
    {
        "prompt": [{"role": "user", "content": row["prompt"]}],
        "completion": [{"role": "assistant", "content": row["kid"]}],
    }
    for row in train_rows
]
train_dataset = Dataset.from_list(train_examples)
assert len(train_dataset) == 1489
assert train_dataset.column_names == ["prompt", "completion"]
assert "adult" not in train_dataset.column_names
assert "public" not in train_dataset.column_names
assert PUBLIC_PATH not in [TRAIN_PATH]

print("Training rows:", len(train_dataset))
print("Training columns:", train_dataset.column_names)
print("kid_adult SHA256:", hashlib.sha256(TRAIN_PATH.read_bytes()).hexdigest())
print("Public test remains unopened until after trainer.train().")

## 6. Load the exact Qwen model in NF4 4-bit and attach LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_id"],
    revision=CONFIG["model_revision"],
    use_fast=True,
)
assert tokenizer.chat_template, "The official Qwen chat template is required"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_id"],
    revision=CONFIG["model_revision"],
    quantization_config=quantization_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
assert getattr(model, "is_loaded_in_4bit", False), "Model was not loaded in 4-bit"
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=CONFIG["lora_target_modules"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
assert model.peft_config["default"].target_modules == set(CONFIG["lora_target_modules"])

## 7. Supervised fine-tuning with completion-only loss

TRL receives a conversational prompt-completion dataset and the official tokenizer chat
template. `completion_only_loss=True` masks the user prompt; because each completion contains
only one assistant message, no assistant-template prompt engineering is added.

In [ ]:
import tempfile

from trl import SFTConfig, SFTTrainer

runtime_root = Path("/content") if Path("/content").is_dir() else Path(tempfile.gettempdir())
training_output_dir = runtime_root / "task1_sft_training"
adapter_output_dir = runtime_root / "task1_sft_adapter"

sft_args = SFTConfig(
    output_dir=str(training_output_dir),
    num_train_epochs=CONFIG["num_train_epochs"],
    learning_rate=CONFIG["learning_rate"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    max_length=CONFIG["max_length"],
    fp16=True,
    bf16=False,
    optim=CONFIG["optim"],
    warmup_ratio=CONFIG["warmup_ratio"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"],
    weight_decay=CONFIG["weight_decay"],
    max_grad_norm=CONFIG["max_grad_norm"],
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    packing=False,
    completion_only_loss=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    seed=SEED,
    data_seed=SEED,
    dataset_num_proc=1,
)
assert sft_args.seed == 42 and sft_args.data_seed == 42
assert sft_args.completion_only_loss is True
assert sft_args.packing is False

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)
train_result = trainer.train()
print("Training metrics:", train_result.metrics)
assert train_result.global_step > 0

## 8. Save only this run's adapter to temporary runtime storage

In [ ]:
model.save_pretrained(adapter_output_dir, safe_serialization=True)
tokenizer.save_pretrained(adapter_output_dir)
assert (adapter_output_dir / "adapter_config.json").is_file()
assert (adapter_output_dir / "adapter_model.safetensors").is_file()
print("Adapter saved inside this runtime:", adapter_output_dir)
print("No adapter was downloaded or loaded from an external source.")